# Lab 17 · Scientific data triage at the terminal

Use Bash to inspect synthetic scientific files, create an auditable set of derived results and preserve the raw inputs. Work through the six tasks, run the checks, then write a short report. No network or third-party Python packages are needed.

On Colab, first save a copy to your Drive. Locally, use a Python kernel with Bash available (Linux/WSL or Git Bash on Windows). Commands in `run_bash(r'''...''')` are real shell commands. Each call starts a fresh Bash process in the same temporary workspace; files persist between calls, shell variables and directory changes do not.

In [ ]:
from pathlib import Path
import csv
import hashlib
import os
import shutil
import subprocess
import tempfile

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False
if in_colab:
    print("Colab runtime: save a notebook copy and download results before the runtime ends.")
else:
    print("Local runtime: keep Bash available and save the completed notebook.")

git_bash = Path(os.environ.get("ProgramFiles", "C:/Program Files")) / "Git/bin/bash.exe"
bash = str(git_bash) if os.name == "nt" and git_bash.is_file() else shutil.which("bash")
if not bash:
    raise RuntimeError("Bash is required: use Linux/WSL, Git Bash on Windows, or Colab.")
workspace = Path(tempfile.mkdtemp(prefix="mm-text-lab-"))
for directory in ("data/raw", "results", "logs"):
    (workspace / directory).mkdir(parents=True, exist_ok=True)

def run_bash(script):
    result = subprocess.run([bash, "-c", script], cwd=workspace,
                            text=True, capture_output=True, check=True,
                            env={**os.environ, "LC_ALL": "C"})
    print(result.stdout, end="")
    return result.stdout

print("Workspace:", workspace)

In [ ]:
fixtures = {
    "samples.tsv": "sample_id\tcondition\treads_million\tqc_status\nS01\tcontrol\t12.4\tpass\nS02\ttreated\t18.2\tpass\nS03\tcontrol\t4.1\tfail\nS04\ttreated\t22.0\tpass\nS05\tcontrol\t15.7\tpass\nS06\ttreated\t7.3\tfail\n",
    "compounds.tsv": "compound_id\tsmiles\tmol_weight\tlogS\tsplit\nC001\tCCO\t46.07\t-0.3\ttrain\nC002\tCC(=O)O\t60.05\t-0.7\ttrain\nC003\tc1ccccc1\t78.11\t-2.1\ttest\nC004\tCC(=O)Oc1ccccc1C(=O)O\t180.16\t-2.3\ttrain\nC005\tCn1c(=O)c2c(ncn2C)n(C)c1=O\t194.19\t-0.1\ttest\nC006\tCCCCCCCC\t114.23\t-4.0\ttrain\n",
    "sequences.fasta": ">seq01 control\nACGTACGT\nACGT\n>seq02 treated\nGGGCCC\nAATT\n>seq03 control\nATATATAT\n",
    "pipeline.log": "INFO start run=demo\nINFO samples=6\nWARN low_reads sample=S03\nWARN low_reads sample=S06\nERROR missing_reference sample=S06\nINFO finished stage=inspect\n",
}
for name, content in fixtures.items():
    (workspace / "data/raw" / name).write_text(content, encoding="utf-8", newline="\n")
raw_hashes = {name: hashlib.sha256((workspace / "data/raw" / name).read_bytes()).hexdigest()
              for name in fixtures}
print("Four synthetic fixtures ready. Numerical values are for teaching, not scientific inference.")

## Task 1 · Count the right records

Set `counts` to the string returned by a Bash snippet that prints the sample-table **line count**, then the number of FASTA headers. Use `wc -l < ...` and anchored `grep -c`. Explain in your report why these numbers count different things. The sample table is newline-terminated and has one header.

In [ ]:
# Set counts using wc -l and grep -c on the fixtures.
# TODO: your code here

In [ ]:
assert "counts" in globals(), "Run your counting task first and assign its output to counts."
assert counts.split() == ["7", "3"], "Expect seven TSV lines and three FASTA headers."
print("✅ Counts verified: six samples plus a header, and three FASTA records.")

## Task 2 · Inspect warnings and errors

Use an anchored extended regular expression to write lines beginning with `WARN ` or `ERROR ` to `results/attention.txt`. Print the file so you can inspect the identifiers. A count of log lines is not a count of distinct samples.

In [ ]:
# Write WARN and ERROR lines into results/attention.txt, then print them.
# TODO: your code here

In [ ]:
attention_path = workspace / "results/attention.txt"
assert attention_path.exists(), "Create results/attention.txt with grep."
attention_lines = attention_path.read_text().splitlines()
assert attention_lines == fixtures["pipeline.log"].splitlines()[2:5], "Keep exactly the two warnings and one error."
print("✅ Three attention lines retained; S06 appears twice.")

## Task 3 · Keep passing samples and their header

Use `awk -F '\t'` to write `results/passing-samples.tsv`. Preserve the first record and then select rows whose fourth field is exactly `pass`. Keep all columns attached to each sample; do not paste separately sorted columns back together.

In [ ]:
# Write the header and passing sample rows to results/passing-samples.tsv.
# TODO: your code here

In [ ]:
passing_path = workspace / "results/passing-samples.tsv"
assert passing_path.exists(), "Create results/passing-samples.tsv with awk."
with passing_path.open(newline="") as handle:
    passing_rows = list(csv.DictReader(handle, delimiter="\t"))
assert [row["sample_id"] for row in passing_rows] == ["S01", "S02", "S04", "S05"], "Check the QC field and preserve sample order."
assert all(row["qc_status"] == "pass" for row in passing_rows), "Every retained row should pass QC."
assert passing_rows[1]["reads_million"] == "18.2", "Keep measurements attached to their sample identifiers."
print("✅ Four passing samples retained, with their header and measurements.")

## Task 4 · Summarise by condition

Write `results/condition-summary.tsv` with no header, in alphabetical condition order. Its columns should be condition, count, and mean reads in millions rounded to two decimal places. Include **all six samples** in this summary. Accumulate arrays in awk, skip the header, format with `printf`, then sort the result.

The fixtures have already been specified as non-negative decimals. When adapting this command to a new dataset, validate field counts and numeric representations first, as in episode 17.2.

In [ ]:
# Write a stable, three-column summary of all six samples.
# TODO: your code here

In [ ]:
summary_path = workspace / "results/condition-summary.tsv"
assert summary_path.exists(), "Write results/condition-summary.tsv."
with summary_path.open(newline="") as handle:
    summary_rows = list(csv.reader(handle, delimiter="\t"))
assert summary_rows == [["control", "3", "10.73"], ["treated", "3", "15.83"]], "Check header skipping, grouping, count and two-decimal formatting."
print("✅ Summary verified; means are reads in millions and include the failed-QC rows.")

## Task 5 · Prepare a molecular subset

Write `results/small-training-compounds.tsv`: header plus training compounds with numeric `mol_weight` below 150 g/mol. Require five fields and a valid non-negative decimal representation before comparing. Preserve the original columns and split labels. This is a mechanical selection exercise on synthetic numerical data, not a recommendation for constructing a test set.

In [ ]:
# Keep the header and training compounds below 150 g/mol.
# TODO: your code here

In [ ]:
selected_path = workspace / "results/small-training-compounds.tsv"
assert selected_path.exists(), "Create the molecular subset with awk."
with selected_path.open(newline="") as handle:
    selected_rows = list(csv.DictReader(handle, delimiter="\t"))
assert [row["compound_id"] for row in selected_rows] == ["C001", "C002", "C006"], "Select by both mass and split."
assert all(row["split"] == "train" for row in selected_rows), "Do not mix test rows into this training subset."
assert selected_rows[1]["smiles"] == "CC(=O)O", "Keep molecular strings unchanged."
print("✅ Three training compounds selected; identifiers and SMILES remain attached.")

## Task 6 · Make a reporting copy, then verify preservation

Use sed to write `results/reporting.log`, changing only a leading `WARN ` into `WARNING `. Print the result. Do not use in-place editing or redirect onto an input. The final check compares the raw file hashes with those captured during setup.

In [ ]:
# Create a derived reporting log with only the leading WARN label expanded.
# TODO: your code here

In [ ]:
report_path = workspace / "results/reporting.log"
assert report_path.exists(), "Create results/reporting.log using sed."
expected_report = "\n".join(
    "WARNING " + line[5:] if line.startswith("WARN ") else line
    for line in fixtures["pipeline.log"].splitlines()
) + "\n"
assert report_path.read_text() == expected_report, "Change only the leading WARN label."
current_hashes = {name: hashlib.sha256((workspace / "data/raw" / name).read_bytes()).hexdigest()
                  for name in fixtures}
assert current_hashes == raw_hashes, "A raw input changed: restart setup and write only derived outputs."
print("✅ Reporting copy verified; every raw fixture is unchanged.")

## Your triage report

Write a short paragraph and an exclusion table in your own words. State the number of sample records, passing samples and FASTA records; distinguish three attention **lines** from distinct affected samples; explain whether the read-depth summary included failed-QC samples. Name one property that these commands do not validate.

Reflection questions:

1. Why would searching `S01` as a substring be unsafe if an identifier `S010` appeared?
2. Why can a FASTA motif search miss a motif that crosses a wrapped line?
3. Why is `awk -F ','` insufficient for arbitrary CSV?
4. How could choosing a preprocessing threshold after inspecting test labels introduce leakage?

Restart the kernel and run all cells to verify reproducibility. Copy any results you need from the printed temporary workspace before ending your session. A new setup run creates a separate workspace.